# RX Strategist OCR Demo

Convert a prescription image to text with Gemini Vision OCR, extract a structured prescription, then verify it with the deterministic engine.

Add `GEMINI_API_KEY` to Colab Secrets before running.

In [ ]:
import sys
from pathlib import Path

REPO_URL = "https://github.com/aravindpj/rx-strategist-mvp.git"

def find_src():
    for path in (Path("src"), Path("rx-strategist-mvp/src"), Path("../src")):
        if (path / "rx_strategist").is_dir():
            return path.resolve()
    return None

src = find_src()
if src is None:
    get_ipython().system(f"git clone {REPO_URL}")
    src = find_src()

if src is None:
    raise FileNotFoundError(
        "Could not find src/rx_strategist. "
        "Run this notebook from the repo folder, or clone "
        "https://github.com/aravindpj/rx-strategist-mvp.git"
    )

sys.path.insert(0, str(src))
req = src.parent / "requirements.txt"
get_ipython().run_line_magic("pip", f"install -q -r {req}")
print("Using", src)

In [ ]:
import sys
from pathlib import Path

for _src in (Path("src"), Path("rx-strategist-mvp/src"), Path("../src")):
    if (_src / "rx_strategist").is_dir():
        sys.path.insert(0, str(_src.resolve()))
        break
else:
    raise FileNotFoundError(
        "Could not find rx_strategist. Run the previous setup cell first."
    )

from google.colab import userdata
from rx_strategist.extraction.gemini_extractor import GeminiPrescriptionExtractor
from rx_strategist.ocr.gemini_ocr import GeminiPrescriptionOCR
from rx_strategist.verification.verifier import verify_prescription

api_key = userdata.get("GEMINI_API_KEY")
ocr = GeminiPrescriptionOCR(api_key=api_key)
extractor = GeminiPrescriptionExtractor(api_key=api_key)

SAMPLE_CANDIDATES = [
    Path("rx-strategist-mvp/data/prescriptions/sample_prescription.png"),
    Path("data/prescriptions/sample_prescription.png"),
    Path("../data/prescriptions/sample_prescription.png"),
]
SAMPLE_IMAGE = next(path for path in SAMPLE_CANDIDATES if path.is_file())
SAMPLE_IMAGE

In [ ]:
raw_prescription = ocr.ocr_image(SAMPLE_IMAGE)
raw_prescription

In [ ]:
# Optional: upload your own prescription photo instead of the sample image.
from google.colab import files

uploaded = files.upload()
if uploaded:
    filename, image_bytes = next(iter(uploaded.items()))
    suffix = Path(filename).suffix.lower()
    mime_types = {
        ".png": "image/png",
        ".jpg": "image/jpeg",
        ".jpeg": "image/jpeg",
        ".webp": "image/webp",
    }
    mime_type = mime_types.get(suffix)
    if not mime_type:
        raise ValueError(f"Unsupported image type: {suffix}")
    raw_prescription = ocr.ocr_image(image_bytes, mime_type=mime_type)
    raw_prescription

In [ ]:
structured_prescription = extractor.extract_prescription(raw_prescription)
structured_prescription

In [ ]:
result = verify_prescription(structured_prescription)
result